<a href="https://colab.research.google.com/github/najeemadelakun/MSC501-M5-Distributed-Processing/blob/main/MSC501_M5_Distributed_Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ============================================================
# MSC501 M5.5: DISTRIBUTED PROCESSING SUBSYSTEM
# STAGE 1: ENVIRONMENT AND PROJECT INITIALISATION
# ============================================================

from pathlib import Path
import subprocess
import sys
import os

# 1. GitHub repository
REPOSITORY_URL = (
    "https://github.com/najeemadelakun/"
    "MSC501-M5-Distributed-Processing.git"
)

PROJECT_NAME = "MSC501-M5-Distributed-Processing"

# 2. Clone repository only if it does not already exist
PROJECT_DIR = Path("/content") / PROJECT_NAME

if not PROJECT_DIR.exists():
    subprocess.run(
        ["git", "clone", REPOSITORY_URL, str(PROJECT_DIR)],
        check=True
    )
else:
    print("Existing project directory detected.")

os.chdir(PROJECT_DIR)

# 3. Create project directories
DIRECTORIES = [
    "src",
    "tests",
    "benchmarks",
    "diagrams",
    "results",
    "reports"
]

for directory in DIRECTORIES:
    (PROJECT_DIR / directory).mkdir(
        parents=True,
        exist_ok=True
    )

# 4. Define project dependencies
REQUIREMENTS = """
numpy
pandas
matplotlib
scipy
pytest
pytest-cov
hypothesis
"""

(PROJECT_DIR / "requirements.txt").write_text(
    REQUIREMENTS.strip() + "\n",
    encoding="utf-8"
)

# 5. Install dependencies once
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        "requirements.txt"
    ],
    check=True
)

# 6. Verify installed dependencies
import numpy
import pandas
import matplotlib
import scipy
import pytest
import hypothesis

# 7. Display project configuration
print("\n" + "=" * 65)
print("MSC501 M5.5 PROJECT INITIALISATION")
print("=" * 65)

print(f"Project directory: {PROJECT_DIR}")
print(f"Python version: {sys.version.split()[0]}")
print(f"NumPy: {numpy.__version__}")
print(f"Pandas: {pandas.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"SciPy: {scipy.__version__}")
print(f"Pytest: {pytest.__version__}")

print("\nPROJECT DIRECTORIES")
for directory in DIRECTORIES:
    print(f"[PASS] {directory}/")

print("\n[PASS] GitHub repository cloned or existing directory reused.")
print("[PASS] Project directories created.")
print("[PASS] Dependencies installed and imported.")
print("[PASS] Environment initialisation completed.")
print("=" * 65)



MSC501 M5.5 PROJECT INITIALISATION
Project directory: /content/MSC501-M5-Distributed-Processing
Python version: 3.13.18
NumPy: 2.4.6
Pandas: 2.2.3
Matplotlib: 3.10.0
SciPy: 1.16.3
Pytest: 8.4.2

PROJECT DIRECTORIES
[PASS] src/
[PASS] tests/
[PASS] benchmarks/
[PASS] diagrams/
[PASS] results/
[PASS] reports/

[PASS] GitHub repository cloned or existing directory reused.
[PASS] Project directories created.
[PASS] Dependencies installed and imported.
[PASS] Environment initialisation completed.


In [2]:

# ============================================================
# MSC501 M5.5
# STAGE 2: CONFIGURATION, DATA MODELS AND INPUT VALIDATION
# ============================================================

from pathlib import Path
import sys
import subprocess
import textwrap

PROJECT_DIR = Path("/content/MSC501-M5-Distributed-Processing")
SRC_DIR = PROJECT_DIR / "src"
TEST_DIR = PROJECT_DIR / "tests"

sys.path.insert(0, str(PROJECT_DIR))

# ------------------------------------------------------------
# 1. SYSTEM CONFIGURATION
# ------------------------------------------------------------

(SRC_DIR / "config.py").write_text(textwrap.dedent("""
    from dataclasses import dataclass

    @dataclass(frozen=True)
    class SystemConfig:
        max_workers: int = 8
        queue_capacity: int = 10000
        max_retries: int = 3
        packet_drop_rate: float = 0.30
        random_seed: int = 42
        target_rate: int = 10000
        database_path: str = "results/transactions.db"
        telemetry_path: str = "results/telemetry.jsonl"

        def __post_init__(self):
            if self.max_workers < 1:
                raise ValueError("max_workers must be positive")
            if self.queue_capacity < 1:
                raise ValueError("queue_capacity must be positive")
            if self.max_retries < 0:
                raise ValueError("max_retries cannot be negative")
            if not 0 <= self.packet_drop_rate <= 1:
                raise ValueError("packet_drop_rate must be between 0 and 1")
            if self.target_rate < 1:
                raise ValueError("target_rate must be positive")

    DEFAULT_CONFIG = SystemConfig()
""").strip() + "\\n", encoding="utf-8")


# ------------------------------------------------------------
# 2. STANDARDISED DATA MODELS
# ------------------------------------------------------------

(SRC_DIR / "models.py").write_text(textwrap.dedent("""
    from dataclasses import dataclass, field
    from datetime import datetime, timezone
    from typing import Any
    from uuid import uuid4

    def utc_timestamp():
        return (
            datetime.now(timezone.utc)
            .isoformat(timespec="microseconds")
            .replace("+00:00", "Z")
        )

    @dataclass(frozen=True)
    class ProcessingRequest:
        request_id: str
        trace_id: str
        payload: dict[str, Any]
        created_at: str = field(default_factory=utc_timestamp)

        @classmethod
        def create(
            cls,
            payload,
            request_id=None,
            trace_id=None,
            created_at=None
        ):
            return cls(
                request_id=request_id or str(uuid4()),
                trace_id=trace_id or str(uuid4()),
                payload=dict(payload),
                created_at=created_at or utc_timestamp()
            )

    @dataclass(frozen=True)
    class ProcessingResult:
        request_id: str
        trace_id: str
        status: str
        result: dict[str, Any] | None
        latency_us: float
        attempts: int
        completed_at: str
        error: str | None = None
""").strip() + "\\n", encoding="utf-8")


# ------------------------------------------------------------
# 3. REQUEST VALIDATION
# ------------------------------------------------------------

(SRC_DIR / "validator.py").write_text(textwrap.dedent("""
    import json
    from datetime import datetime

    def validate_request(request):
        required_ids = {
            "request_id": request.request_id,
            "trace_id": request.trace_id
        }

        for field, value in required_ids.items():
            if not isinstance(value, str) or not value.strip():
                raise ValueError(f"{field} must be a non-empty string")

        if not isinstance(request.payload, dict):
            raise ValueError("payload must be a dictionary")

        try:
            json.dumps(request.payload, allow_nan=False)
        except (TypeError, ValueError) as exc:
            raise ValueError("payload must contain valid JSON data") from exc

        if not isinstance(request.created_at, str):
            raise ValueError("created_at must be an ISO-8601 string")

        try:
            parsed = datetime.fromisoformat(
                request.created_at.replace("Z", "+00:00")
            )
        except ValueError as exc:
            raise ValueError("created_at must be a valid ISO-8601 timestamp") from exc

        if parsed.tzinfo is None or parsed.utcoffset() is None:
            raise ValueError("created_at must include a timezone")

        return True
""").strip() + "\\n", encoding="utf-8")


# ------------------------------------------------------------
# 4. PACKAGE INITIALISATION
# ------------------------------------------------------------

(SRC_DIR / "__init__.py").write_text(
    '"""MSC501 M5.5 distributed processing subsystem."""\\n',
    encoding="utf-8"
)


# ------------------------------------------------------------
# 5. AUTOMATED TESTS
# ------------------------------------------------------------

(TEST_DIR / "test_stage2.py").write_text(textwrap.dedent("""
    import pytest

    from src.config import SystemConfig, DEFAULT_CONFIG
    from src.models import ProcessingRequest, ProcessingResult
    from src.validator import validate_request

    def test_default_configuration():
        assert DEFAULT_CONFIG.max_workers == 8
        assert DEFAULT_CONFIG.packet_drop_rate == 0.30
        assert DEFAULT_CONFIG.target_rate == 10000

    def test_invalid_configuration():
        with pytest.raises(ValueError):
            SystemConfig(max_workers=0)

        with pytest.raises(ValueError):
            SystemConfig(packet_drop_rate=1.5)

    def test_request_creation():
        request = ProcessingRequest.create({"value": 25})
        assert request.request_id
        assert request.trace_id
        assert request.payload["value"] == 25

    def test_valid_request():
        request = ProcessingRequest.create(
            {"operation": "addition", "value": 25}
        )
        assert validate_request(request) is True

    def test_empty_request_id_rejected():
        request = ProcessingRequest(
            request_id="",
            trace_id="trace-001",
            payload={"value": 1}
        )
        with pytest.raises(ValueError):
            validate_request(request)

    def test_invalid_payload_rejected():
        request = ProcessingRequest.create({"value": float("nan")})
        with pytest.raises(ValueError):
            validate_request(request)

    def test_timezone_required():
        request = ProcessingRequest(
            request_id="req-001",
            trace_id="trace-001",
            payload={"value": 1},
            created_at="2026-11-08T10:00:00"
        )
        with pytest.raises(ValueError):
            validate_request(request)

    def test_processing_result_structure():
        result = ProcessingResult(
            request_id="req-001",
            trace_id="trace-001",
            status="SUCCESS",
            result={"value": 25},
            latency_us=100.0,
            attempts=1,
            completed_at="2026-11-08T10:00:00Z"
        )
        assert result.status == "SUCCESS"
        assert result.attempts == 1
""").strip() + "\\n", encoding="utf-8")


# ------------------------------------------------------------
# 6. IMPORT AND TEST VERIFICATION
# ------------------------------------------------------------

from src.config import SystemConfig, DEFAULT_CONFIG
from src.models import ProcessingRequest, ProcessingResult
from src.validator import validate_request

print("=" * 65)
print("MSC501 M5.5 STAGE 2: MODULE VERIFICATION")
print("=" * 65)

for filename in ["config.py", "models.py", "validator.py", "__init__.py"]:
    assert (SRC_DIR / filename).exists()
    print(f"[PASS] src/{filename}")

test_run = subprocess.run(
    [
        sys.executable, "-m", "pytest",
        "tests/test_stage2.py", "-v", "--tb=short"
    ],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True
)

print("\nAUTOMATED TEST RESULTS")
print(test_run.stdout)

if test_run.returncode != 0:
    print(test_run.stderr)
    raise RuntimeError("Stage 2 tests failed.")

print("[PASS] All Stage 2 tests completed successfully.")
print("=" * 65)


SyntaxError: unexpected character after line continuation character (__init__.py, line 1)